# HuggingFace 커스텀 프로젝트
## 1. 환경 설정

- `transformers` : HuggingFace 핵심 라이브러리
- `accelerate` : 분산학습 및 GPU 최적화 지원

In [1]:
# transformers 설치
!pip install transformers

# accelerate 설치 (GPU 학습 최적화)
!pip install accelerate

In [2]:
# transformers 설치 확인
# pipeline으로 감성분석 테스트
# 'I love you' 문장이 POSITIVE인지 확인
!python -c "from transformers import pipeline; print(pipeline('sentiment-analysis')('I love you'))"

[transformers] No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.
Loading weights: 100%|██████████████████████| 104/104 [00:00<00:00, 3344.41it/s]
[{'label': 'POSITIVE', 'score': 0.9998656511306763}]


## 2. 커스텀 프로젝트 제작
### (1) Datasets - HuggingFace에서 불러오기

- `datasets` : HuggingFace 데이터셋 라이브러리
- GLUE, SQuAD 등 다양한 공개 데이터셋 바로 사용 가능
- 한번 다운로드 후 자동 캐싱

In [3]:
# HuggingFace datasets 라이브러리 설치
!pip install datasets

### GLUE MRPC 데이터셋 로드

- `glue` : GLUE benchmark 데이터셋
- `mrpc` : 두 문장의 유사도 평가 task
    - label 0 = 다른 의미 / label 1 = 같은 의미
- train / validation / test 3가지 split 제공

In [4]:
import datasets
from datasets import load_dataset

# GLUE benchmark에서 MRPC 데이터셋 로드
# mrpc : Microsoft Research Paraphrase Corpus
# 두 문장이 같은 의미인지 판단하는 task (label 0=다름, 1=같음)
huggingface_mrpc_dataset = load_dataset('glue', 'mrpc')
print(huggingface_mrpc_dataset)

DatasetDict({
    train: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 3668
    })
    validation: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 408
    })
    test: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 1725
    })
})


### 데이터셋 컬럼 확인

- train 데이터셋의 컬럼명 확인
- `sentence1`, `sentence2`, `label`, `idx` 4가지 컬럼

In [5]:
# train 데이터셋 꺼내기
train = huggingface_mrpc_dataset['train']

# 컬럼명 확인
cols = train.column_names
cols

['sentence1', 'sentence2', 'label', 'idx']

### 데이터 샘플 확인

- 첫 5개 샘플 출력
- sentence1, sentence2 두 문장 쌍과 label 확인

In [6]:
# 첫 5개 샘플 출력
# 각 샘플의 sentence1, sentence2, label, idx 확인
for i in range(5):
    for col in cols:
        print(col, ":", train[col][i])
    print('\n')

sentence1 : Amrozi accused his brother , whom he called " the witness " , of deliberately distorting his evidence .
sentence2 : Referring to him as only " the witness " , Amrozi accused his brother of deliberately distorting his evidence .
label : 1
idx : 0


sentence1 : Yucaipa owned Dominick 's before selling the chain to Safeway in 1998 for $ 2.5 billion .
sentence2 : Yucaipa bought Dominick 's in 1995 for $ 693 million and sold it to Safeway for $ 1.8 billion in 1998 .
label : 0
idx : 1


sentence1 : They had published an advertisement on the Internet on June 10 , offering the cargo for sale , he added .
sentence2 : On June 10 , the ship 's owners had published an advertisement on the Internet , offering the explosives for sale .
label : 1
idx : 2


sentence1 : Around 0335 GMT , Tab shares were up 19 cents , or 4.4 % , at A $ 4.56 , having earlier set a record high of A $ 4.57 .
sentence2 : Tab shares jumped 20 cents , or 4.6 % , to set a record closing high at A $ 4.57 .
label : 0

### 커스텀 데이터셋 만들기

- MRPC 원본 파일을 직접 파싱해서 HuggingFace Dataset 형태로 변환
- 원본 파일 컬럼: `Quality`, `#1 ID`, `#2 ID`, `#1 String`, `#2 String`
- HuggingFace MRPC와 다른 점:
    - `sentence1` → `#1 String`
    - `sentence2` → `#2 String`
    - `label` → `Quality`

In [9]:
import pandas as pd
from datasets import Dataset

def parse_mrpc_file(file_path):
    """MRPC 파일을 안전하게 파싱하는 함수"""
    try:
        df = pd.read_csv(file_path, sep='\t', on_bad_lines='skip')
        print(f"원본 컬럼: {df.columns.tolist()}")
        
        if 'Quality' not in df.columns and 'label' in df.columns:
            df = df.rename(columns={'label': 'Quality'})
        if '#1 String' not in df.columns and 'sentence1' in df.columns:
            df = df.rename(columns={'sentence1': '#1 String'})
        if '#2 String' not in df.columns and 'sentence2' in df.columns:
            df = df.rename(columns={'sentence2': '#2 String'})
        if '#1 ID' not in df.columns:
            df['#1 ID'] = range(len(df))
        if '#2 ID' not in df.columns:
            df['#2 ID'] = range(len(df))

        df = df[['Quality', '#1 ID', '#2 ID', '#1 String', '#2 String']]
        return df

    except Exception as e:
        print(f"Error: {e}")
        return pd.DataFrame()

# 로컬 파일에서 MRPC 데이터 읽기
train_df = parse_mrpc_file('data/msr_paraphrase_train.txt')
test_df = parse_mrpc_file('data/msr_paraphrase_test.txt')

print(f"train: {len(train_df)}개, test: {len(test_df)}개")
print(train_df.head())

원본 컬럼: ['#1 String', '#2 String', 'Quality', 'idx']
원본 컬럼: ['#1 String', '#2 String', 'Quality', 'idx']
train: 3668개, test: 1725개
   Quality  #1 ID  #2 ID                                          #1 String  \
0        1      0      0  Amrozi accused his brother , whom he called " ...   
1        0      1      1  Yucaipa owned Dominick 's before selling the c...   
2        1      2      2  They had published an advertisement on the Int...   
3        0      3      3  Around 0335 GMT , Tab shares were up 19 cents ...   
4        1      4      4  The stock rose $ 2.11 , or about 11 percent , ...   

                                           #2 String  
0  Referring to him as only " the witness " , Amr...  
1  Yucaipa bought Dominick 's in 1995 for $ 693 m...  
2  On June 10 , the ship 's owners had published ...  
3  Tab shares jumped 20 cents , or 4.6 % , to set...  
4  PG & E Corp. shares jumped $ 1.63 or 8 percent...  


### 데이터 구조 상세 확인

- 컬럼명, 데이터 shape, 샘플 5개 출력
- HuggingFace MRPC vs 커스텀 MRPC 컬럼 비교
    - HF: `sentence1`, `sentence2`, `label`
    - 커스텀: `#1 String`, `#2 String`, `Quality`

In [10]:
# 데이터 구조 확인
print("Train dataset columns:", train_df.columns.tolist())
print("Train dataset shape:", train_df.shape)

print("\nFirst 5 examples:")
for i in range(5):
    row = train_df.iloc[i]
    for col in train_df.columns:
        print(f"{col}: {row[col]}")
    print('\n')

Train dataset columns: ['Quality', '#1 ID', '#2 ID', '#1 String', '#2 String']
Train dataset shape: (3668, 5)

First 5 examples:
Quality: 1
#1 ID: 0
#2 ID: 0
#1 String: Amrozi accused his brother , whom he called " the witness " , of deliberately distorting his evidence .
#2 String: Referring to him as only " the witness " , Amrozi accused his brother of deliberately distorting his evidence .


Quality: 0
#1 ID: 1
#2 ID: 1
#1 String: Yucaipa owned Dominick 's before selling the chain to Safeway in 1998 for $ 2.5 billion .
#2 String: Yucaipa bought Dominick 's in 1995 for $ 693 million and sold it to Safeway for $ 1.8 billion in 1998 .


Quality: 1
#1 ID: 2
#2 ID: 2
#1 String: They had published an advertisement on the Internet on June 10 , offering the cargo for sale , he added .
#2 String: On June 10 , the ship 's owners had published an advertisement on the Internet , offering the explosives for sale .


Quality: 0
#1 ID: 3
#2 ID: 3
#1 String: Around 0335 GMT , Tab shares were up 19 

### 커스텀 DatasetDict 생성

- train(80%) / validation(20%) / test 분할
- `stratify=Quality` → 라벨 비율 유지하며 분할 (데이터 불균형 방지)
- HuggingFace 표준 DatasetDict 형태로 변환

In [11]:
from datasets import Dataset, DatasetDict
from sklearn.model_selection import train_test_split

# DataFrame을 dict 형식으로 변경
train_dataset = train_df.to_dict('list')
test_dataset = test_df.to_dict('list')

# train 데이터를 train(80%)과 validation(20%)으로 분할
# stratify=Quality → 라벨 비율 유지
train_indices = list(range(len(train_dataset['Quality'])))
train_idx, val_idx = train_test_split(
    train_indices,
    test_size=0.2,
    random_state=42,
    stratify=train_dataset['Quality']
)

# validation 데이터셋 생성
validation_dataset = {}
for key in train_dataset.keys():
    validation_dataset[key] = [train_dataset[key][i] for i in val_idx]

# train 데이터셋 업데이트 (validation으로 사용된 데이터 제거)
train_dataset_final = {}
for key in train_dataset.keys():
    train_dataset_final[key] = [train_dataset[key][i] for i in train_idx]

# HuggingFace Dataset 객체로 변환
train_hf_dataset = Dataset.from_dict(train_dataset_final)
validation_hf_dataset = Dataset.from_dict(validation_dataset)
test_hf_dataset = Dataset.from_dict(test_dataset)

# DatasetDict 생성 (HuggingFace 표준 방식)
customized_mrpc_dataset = DatasetDict({
    'train': train_hf_dataset,
    'validation': validation_hf_dataset,
    'test': test_hf_dataset
})

# 결과 출력
print("DatasetDict({")
for split_name, split_data in customized_mrpc_dataset.items():
    print(f"    {split_name}: Dataset({{")
    print(f"        features: {list(split_data.features.keys())},")
    print(f"        num_rows: {split_data.num_rows}")
    print("    })")
print("})")

print(f"\nTrain     : {customized_mrpc_dataset['train'].num_rows}개")
print(f"Validation: {customized_mrpc_dataset['validation'].num_rows}개")
print(f"Test      : {customized_mrpc_dataset['test'].num_rows}개")

# 첫 번째 샘플 확인
print(f"\n첫 번째 train 샘플:")
print(customized_mrpc_dataset['train'][0])

DatasetDict({
    train: Dataset({
        features: ['Quality', '#1 ID', '#2 ID', '#1 String', '#2 String'],
        num_rows: 2934
    })
    validation: Dataset({
        features: ['Quality', '#1 ID', '#2 ID', '#1 String', '#2 String'],
        num_rows: 734
    })
    test: Dataset({
        features: ['Quality', '#1 ID', '#2 ID', '#1 String', '#2 String'],
        num_rows: 1725
    })
})

Train     : 2934개
Validation: 734개
Test      : 1725개

첫 번째 train 샘플:
{'Quality': 0, '#1 ID': 2499, '#2 ID': 2499, '#1 String': 'The indictment supercedes a criminal complaint filed against Quattrone on April 23 with similar charges .', '#2 String': 'The indictment follows a criminal complaint filed by federal prosecutors on April 23 .'}


## (2) Tokenizer와 Model

### HuggingFace Auto Classes 활용
- `AutoTokenizer` : 모델에 맞는 토크나이저 자동 선택
- `AutoModelForSequenceClassification` : 문장 분류용 모델
- `distilbert-base-uncased` : BERT의 경량화 버전
    - BERT 대비 40% 작고 60% 빠름
    - 성능은 BERT의 97% 유지
- `num_labels=2` : MRPC label 2개 (0=다름, 1=같음)

In [12]:
import transformers
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# distilbert-base-uncased : BERT 경량화 버전
# - BERT보다 40% 작고 60% 빠름
# - 성능은 BERT의 97% 유지
huggingface_tokenizer = AutoTokenizer.from_pretrained('distilbert-base-uncased')

# AutoModelForSequenceClassification : 문장 분류용 모델
# num_labels=2 : MRPC label 2개 (0=다름, 1=같음)
huggingface_model = AutoModelForSequenceClassification.from_pretrained(
    'distilbert-base-uncased',
    num_labels=2
)

print(f"토크나이저 : {huggingface_tokenizer.__class__.__name__}")
print(f"모델       : {huggingface_model.__class__.__name__}")

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


토크나이저 : BertTokenizer
모델       : DistilBertForSequenceClassification


### Tokenizer 함수 정의

- `transform` : 데이터셋의 문장 쌍을 토크나이징하는 함수
- HuggingFace MRPC 컬럼명 사용 (`sentence1`, `sentence2`)
- 주요 옵션:
    - `truncation=True` : 최대 길이 초과시 자동으로 잘라냄
    - `padding='max_length'` : 최대 길이에 맞게 패딩 추가
    - `return_token_type_ids=False` : MRPC task에 불필요해서 제거

### Auto Class 장점
- `AutoTokenizer` : BERT→DistilBERT→RoBERTa 모델 변경시 코드 수정 불필요
- `AutoModel` : 모델만 바꿔도 자동으로 알맞은 클래스 선택

In [13]:
# HuggingFace MRPC용 토크나이징 함수
# sentence1, sentence2 두 문장을 동시에 토크나이징
def transform(data):
    return huggingface_tokenizer(
        data['sentence1'],   # 첫번째 문장
        data['sentence2'],   # 두번째 문장
        truncation=True,          # 최대 길이 초과시 자르기
        padding='max_length',     # 최대 길이에 맞게 패딩
        return_token_type_ids=False,  # MRPC task에 불필요
    )

# 전체 데이터셋에 토크나이징 적용
huggingface_mrpc_dataset_tokenized = huggingface_mrpc_dataset.map(
    transform,
    batched=True  # 배치 단위 처리 → 속도 향상
)

print(huggingface_mrpc_dataset_tokenized)
print(f"\n토크나이징 완료!")
print(f"추가된 컬럼: input_ids, attention_mask")

Map:   0%|          | 0/3668 [00:00<?, ? examples/s]

Map:   0%|          | 0/408 [00:00<?, ? examples/s]

Map:   0%|          | 0/1725 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx', 'input_ids', 'attention_mask'],
        num_rows: 3668
    })
    validation: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx', 'input_ids', 'attention_mask'],
        num_rows: 408
    })
    test: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx', 'input_ids', 'attention_mask'],
        num_rows: 1725
    })
})

토크나이징 완료!
추가된 컬럼: input_ids, attention_mask


### 전체 데이터셋 토크나이징 (map 사용)

- `map()` : DatasetDict 전체에 함수 일괄 적용
- `batched=True` : 배치 단위로 처리 → 속도 향상
- 토크나이징 후 train / validation / test 분리

In [14]:
# map으로 전체 데이터셋 한번에 토크나이징
# batched=True : 배치 단위 처리 → 속도 향상
hf_dataset = huggingface_mrpc_dataset.map(transform, batched=True)

# train / validation / test split
hf_train_dataset = hf_dataset['train']
hf_val_dataset = hf_dataset['validation']
hf_test_dataset = hf_dataset['test']

# 결과 확인
print(hf_dataset)
print(f"\ntrain      : {len(hf_train_dataset)}개")
print(f"validation : {len(hf_val_dataset)}개")
print(f"test       : {len(hf_test_dataset)}개")
print(f"\n추가된 컬럼: {[c for c in hf_train_dataset.column_names if c not in ['sentence1','sentence2','label','idx']]}")

DatasetDict({
    train: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx', 'input_ids', 'attention_mask'],
        num_rows: 3668
    })
    validation: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx', 'input_ids', 'attention_mask'],
        num_rows: 408
    })
    test: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx', 'input_ids', 'attention_mask'],
        num_rows: 1725
    })
})

train      : 3668개
validation : 408개
test       : 1725개

추가된 컬럼: ['input_ids', 'attention_mask']


### Q. 커스텀 데이터셋에 HF용 transform 함수 적용하면?

- HF MRPC 컬럼명: `sentence1`, `sentence2`
- 커스텀 MRPC 컬럼명: `#1 String`, `#2 String`
- 다른 컬럼명으로 transform 적용시 어떤 오류가 발생하는지 확인!

In [15]:
# Q. custom 데이터셋에 HF용 transform 함수를 매핑하면?
# custom 데이터셋은 sentence1, sentence2가 아닌
# '#1 String', '#2 String' 컬럼명을 사용하기 때문에 오류 발생!
tf_train_dataset_error = customized_mrpc_dataset['train'].map(transform)

Map:   0%|          | 0/2934 [00:00<?, ? examples/s]

KeyError: 'sentence1'

In [16]:
# ⚠️ 데이터는 이미 셀[11]에서 customized_mrpc_dataset으로 만들어놨음
# val_dataset = test_dataset.copy() 절대 하면 안됨 → 데이터 누수!

# 커스텀 데이터용 transform 함수 정의
# HF MRPC와 컬럼명이 다름 (#1 String, #2 String)
def transform_custom(batch):
    return huggingface_tokenizer(
        batch['#1 String'],   # 첫번째 문장
        batch['#2 String'],   # 두번째 문장
        truncation=True,
        padding='max_length',
        return_token_type_ids=False,
    )

# Quality → label 로 rename (HF Trainer가 자동 인식하려면 필수!)
customized_mrpc_dataset = customized_mrpc_dataset.rename_column('Quality', 'label')

# 전체 DatasetDict에 토크나이징 일괄 적용
customized_mrpc_dataset = customized_mrpc_dataset.map(transform_custom, batched=True)

# split 꺼내쓰기
custom_train_dataset = customized_mrpc_dataset['train']
custom_val_dataset = customized_mrpc_dataset['validation']
custom_test_dataset = customized_mrpc_dataset['test']

print('custom dataset 토큰화 완료')
print(f'train: {len(custom_train_dataset)}, val: {len(custom_val_dataset)}, test: {len(custom_test_dataset)}')
print(f'features: {custom_train_dataset.column_names}')

Map:   0%|          | 0/2934 [00:00<?, ? examples/s]

Map:   0%|          | 0/734 [00:00<?, ? examples/s]

Map:   0%|          | 0/1725 [00:00<?, ? examples/s]

custom dataset 토큰화 완료
train: 2934, val: 734, test: 1725
features: ['label', '#1 ID', '#2 ID', '#1 String', '#2 String', 'input_ids', 'attention_mask']


## (3) Train/Evaluation과 Test
### Trainer를 활용한 학습

### TrainingArguments 설정
| 인자 | 값 | 설명 |
|------|-----|------|
| `output_dir` | transformers | 체크포인트 저장 경로 |
| `eval_strategy` | epoch | 매 epoch마다 평가 |
| `learning_rate` | 2e-5 | 학습률 |
| `per_device_train_batch_size` | 8 | 학습 배치 크기 |
| `per_device_eval_batch_size` | 8 | 평가 배치 크기 |
| `num_train_epochs` | 3 | 학습 반복 횟수 |
| `weight_decay` | 0.01 | 과적합 방지 |

In [17]:
import os
import numpy as np
from transformers import Trainer, TrainingArguments

output_dir = 'transformers'

training_arguments = TrainingArguments(
    output_dir,                          # 체크포인트 저장 경로
    eval_strategy="epoch",               # 매 epoch 끝날때마다 평가
    learning_rate=2e-5,                  # 학습률
    per_device_train_batch_size=8,       # 학습 배치 크기
    per_device_eval_batch_size=8,        # 평가 배치 크기
    num_train_epochs=3,                  # 총 학습 epoch 수
    weight_decay=0.01,                   # L2 정규화 → 과적합 방지
)

print(f"output_dir    : {training_arguments.output_dir}")
print(f"eval_strategy : {training_arguments.eval_strategy}")
print(f"learning_rate : {training_arguments.learning_rate}")
print(f"epochs        : {training_arguments.num_train_epochs}")
print(f"batch_size    : {training_arguments.per_device_train_batch_size}")

output_dir    : transformers
eval_strategy : IntervalStrategy.EPOCH
learning_rate : 2e-05
epochs        : 3
batch_size    : 8


### compute_metrics 설정

- `evaluate` 라이브러리 : HuggingFace 평가 지표 라이브러리
- MRPC는 **binary classification** → Accuracy + F1 사용
- `compute_metrics` : Trainer에 전달할 평가 함수
    - 모델 출력(logits) → argmax → 예측값
    - 예측값 vs 실제 label 비교해서 성능 계산

In [18]:
# evaluate 라이브러리 설치
# HuggingFace 공식 평가 지표 라이브러리
!pip install evaluate

### compute_metrics 함수 정의

- `metric = load('glue', 'mrpc')` : MRPC 공식 평가 지표 로드
- `eval_pred` : 모델 출력 (logits, labels) 튜플
- `np.argmax` : logits → 예측 클래스 변환
    - logits: [0.3, 0.7] → argmax → 1 (같은 의미)
- 최종 반환: `accuracy` + `f1`

In [19]:
from evaluate import load

# GLUE MRPC 공식 평가 지표 로드 (accuracy + f1)
metric = load('glue', 'mrpc')

def compute_metrics(eval_pred):
    predictions, labels = eval_pred

    # logits → 예측 클래스 변환
    # ex) [0.3, 0.7] → 1 (같은 의미)
    # ex) [0.8, 0.2] → 0 (다른 의미)
    predictions = np.argmax(predictions, axis=1)

    # accuracy + f1 계산
    return metric.compute(predictions=predictions, references=labels)

print("compute_metrics 함수 정의 완료!")
print("평가 지표: accuracy + f1")

compute_metrics 함수 정의 완료!
평가 지표: accuracy + f1


### HuggingFace MRPC 학습 시작!

- `Trainer` : 학습 전 과정을 관리하는 클래스
- HuggingFace 공식 MRPC 데이터셋으로 학습
- epoch마다 validation 성능 확인

In [20]:
trainer = Trainer(
    model=huggingface_model,            # 학습시킬 모델
    args=training_arguments,            # TrainingArguments 설정값
    train_dataset=hf_train_dataset,     # HF MRPC 학습 데이터
    eval_dataset=hf_val_dataset,        # HF MRPC 검증 데이터
    compute_metrics=compute_metrics,    # accuracy + f1 평가
)

# 학습 시작!
trainer.train()
print("슝~")

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,No log,0.368115,0.825980,0.879046
2,0.506676,0.400012,0.850490,0.896785
3,0.329338,0.576974,0.850490,0.895726


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

슝~


### HuggingFace MRPC Test 평가

- 학습 중 한번도 보지 않은 test 데이터로 최종 평가
- `trainer.evaluate()` : 모델 성능 평가

In [21]:
# test 데이터셋으로 최종 평가
# 학습 중 한번도 본 적 없는 데이터로 실제 성능 측정
trainer.evaluate(hf_test_dataset)

Training Loss,Validation Loss,Epoch,Accuracy,F1
0.329338,0.602678,3,0.829565,0.875212


{'eval_loss': 0.6026781797409058,
 'eval_accuracy': 0.8295652173913044,
 'eval_f1': 0.8752122241086587}

### 메모리 정리

- 학습 완료된 모델 메모리에서 제거
- GPU 메모리 확보 → 다음 커스텀 모델 학습 준비

In [22]:
import gc
import torch

# 학습 완료된 모델 메모리에서 제거
del huggingface_model
del trainer

# 가비지 컬렉터 실행
gc.collect()

# GPU 캐시 비우기
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("메모리 정리 완료!")

메모리 정리 완료!


### 커스텀 데이터셋으로 학습

- 앞서 만든 커스텀 MRPC 데이터셋으로 학습
- HF MRPC와 동일한 모델 구조 사용
- output_dir 분리 → 기존 체크포인트와 충돌 방지

In [23]:
# 커스텀 데이터셋 학습용 모델 새로 로드
# (기존 huggingface_model은 메모리에서 삭제했으므로 새로 로드)
huggingface_model_custom = AutoModelForSequenceClassification.from_pretrained(
    'distilbert-base-uncased',
    num_labels=2
)

# 커스텀 학습용 TrainingArguments 별도 생성
# output_dir 분리 → 기존 HF MRPC 체크포인트와 충돌 방지
training_arguments_custom = TrainingArguments(
    output_dir='transformers_custom',    # ✅ 기존 'transformers'와 분리!
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
)

trainer_custom = Trainer(
    model=huggingface_model_custom,      # ✅ 새 모델 변수
    args=training_arguments_custom,      # ✅ 커스텀 arguments
    train_dataset=custom_train_dataset,  # ✅ tf_train_dataset → custom_train_dataset
    eval_dataset=custom_val_dataset,     # ✅ tf_val_dataset → custom_val_dataset
    compute_metrics=compute_metrics,
)

trainer_custom.train()
print("슝~")

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,No log,0.435019,0.807902,0.859141
2,0.496143,0.443220,0.832425,0.877855
3,0.298570,0.532500,0.831063,0.876984


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

슝~


### 커스텀 데이터셋 Validation 평가

- `trainer_custom.evaluate()` : validation 데이터셋으로 평가
- 학습 중 모니터링한 validation 셋 기준 최종 성능 확인

In [24]:
# Validation 데이터셋 평가
# 학습 중 모니터링한 validation 셋으로 최종 성능 확인
print('\n=== 커스텀 validation 셋 평가 ===')
eval_results = trainer_custom.evaluate()
print(eval_results)

# test 데이터셋 최종 평가
# 학습 중 한번도 보지 않은 데이터로 실제 성능 측정
print('\n=== 커스텀 test 셋 최종 평가 ===')
print(trainer_custom.evaluate(custom_test_dataset))


=== 커스텀 validation 셋 평가 ===


Training Loss,Validation Loss,Epoch,Accuracy,F1
0.298570,0.532500,3,0.831063,0.876984


{'eval_loss': 0.5324997901916504, 'eval_accuracy': 0.8310626702997275, 'eval_f1': 0.876984126984127}

=== 커스텀 test 셋 최종 평가 ===


Training Loss,Validation Loss,Epoch,Accuracy,F1
0.298570,0.546572,3,0.820290,0.868977


{'eval_loss': 0.5465715527534485, 'eval_accuracy': 0.8202898550724638, 'eval_f1': 0.8689771766694844}


# 16. 커스텀 프로젝트 - NSMC 감성분석
## 1. 라이브러리 버전 확인

- `klue/bert-base` : 한국어 특화 BERT 모델
- `NSMC` : 네이버 영화 리뷰 감성분석 데이터셋
    - label 0 = 부정 / label 1 = 긍정

In [26]:
import numpy
import transformers
import datasets

# 라이브러리 버전 확인
print(f"numpy       : {numpy.__version__}")
print(f"transformers: {transformers.__version__}")
print(f"datasets    : {datasets.__version__}")

numpy       : 2.2.6
transformers: 5.9.0
datasets    : 4.8.5


## STEP 1. NSMC 데이터 분석 및 HuggingFace Dataset 구성

### NSMC란?
- **Naver Sentiment Movie Corpus**
- 네이버 영화 리뷰 감성분석 데이터셋
- label 0 = 부정 / label 1 = 긍정
- train: 150,000개 / test: 50,000개

In [29]:
from datasets import load_dataset

# HuggingFace에서 NSMC 데이터셋 로드
nsmc_dataset = load_dataset('e9t/nsmc')
print(nsmc_dataset)

# 데이터 샘플 확인
print("\n=== 샘플 데이터 확인 ===")
for i in range(3):
    print(f"문장 : {nsmc_dataset['train'][i]['document']}")
    print(f"label: {nsmc_dataset['train'][i]['label']}")
    print()

RuntimeError: Dataset scripts are no longer supported, but found nsmc.py

In [30]:
from datasets import load_dataset

# GitHub raw 파일로 직접 로드
nsmc_dataset = load_dataset(
    'csv',
    data_files={
        'train': 'https://raw.githubusercontent.com/e9t/nsmc/master/ratings_train.txt',
        'test': 'https://raw.githubusercontent.com/e9t/nsmc/master/ratings_test.txt'
    },
    delimiter='\t'
)
print(nsmc_dataset)

Generating train split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'document', 'label'],
        num_rows: 150000
    })
    test: Dataset({
        features: ['id', 'document', 'label'],
        num_rows: 50000
    })
})


## STEP 2. klue/bert-base 모델 & 토크나이저 로드

### klue/bert-base란?
- 한국어 특화 BERT 모델
- KLUE 벤치마크 학습에 최적화
- 한국어 형태소 기반 토크나이저 사용

In [31]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# klue/bert-base 토크나이저 로드
# 한국어 특화 토크나이저
klue_tokenizer = AutoTokenizer.from_pretrained('klue/bert-base')

# klue/bert-base 모델 로드
# num_labels=2 : NSMC label 2개 (0=부정, 1=긍정)
klue_model = AutoModelForSequenceClassification.from_pretrained(
    'klue/bert-base',
    num_labels=2
)

print(f"토크나이저 : {klue_tokenizer.__class__.__name__}")
print(f"모델       : {klue_model.__class__.__name__}")

# 토크나이저 테스트
test_sentence = "이 영화 정말 재미있어요!"
tokens = klue_tokenizer.tokenize(test_sentence)
print(f"\n테스트 문장 : {test_sentence}")
print(f"토크나이징  : {tokens}")

config.json:   0%|          | 0.00/425 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/289 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/445M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: klue/bert-base
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


토크나이저 : BertTokenizer
모델       : BertForSequenceClassification

테스트 문장 : 이 영화 정말 재미있어요!
토크나이징  : ['이', '영화', '정말', '재미있', '##어요', '!']


## STEP 3. 데이터 전처리 & 학습

### 진행 순서
1. NSMC 데이터 전처리 (null 제거 등)
2. train/validation split
3. 토크나이징
4. Trainer로 학습
5. 평가 → Validation accuracy 90% 목표!

In [32]:
import numpy as np
from datasets import DatasetDict

# 결측값 제거 (document가 None인 경우)
nsmc_dataset = nsmc_dataset.filter(lambda x: x['document'] is not None)

# train/validation split (train 90%, validation 10%)
train_val = nsmc_dataset['train'].train_test_split(
    test_size=0.1,
    seed=42
)

# DatasetDict 재구성
nsmc_dataset = DatasetDict({
    'train'     : train_val['train'],
    'validation': train_val['test'],
    'test'      : nsmc_dataset['test']
})

print(nsmc_dataset)
print(f"\ntrain     : {len(nsmc_dataset['train'])}개")
print(f"validation: {len(nsmc_dataset['validation'])}개")
print(f"test      : {len(nsmc_dataset['test'])}개")

Filter:   0%|          | 0/150000 [00:00<?, ? examples/s]

Filter:   0%|          | 0/50000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'document', 'label'],
        num_rows: 134995
    })
    validation: Dataset({
        features: ['id', 'document', 'label'],
        num_rows: 15000
    })
    test: Dataset({
        features: ['id', 'document', 'label'],
        num_rows: 49997
    })
})

train     : 134995개
validation: 15000개
test      : 49997개


In [33]:
# NSMC 토크나이징 함수
def tokenize_nsmc(batch):
    return klue_tokenizer(
        batch['document'],    # 리뷰 텍스트
        truncation=True,      # 최대 길이 초과시 자르기
        padding='max_length', # 최대 길이에 맞게 패딩
        max_length=128,       # NSMC는 짧은 문장 많아서 128로 설정
        return_token_type_ids=False,
    )

# 전체 데이터셋 토크나이징
nsmc_tokenized = nsmc_dataset.map(tokenize_nsmc, batched=True)

print("토크나이징 완료!")
print(nsmc_tokenized)

Map:   0%|          | 0/134995 [00:00<?, ? examples/s]

Map:   0%|          | 0/15000 [00:00<?, ? examples/s]

Map:   0%|          | 0/49997 [00:00<?, ? examples/s]

토크나이징 완료!
DatasetDict({
    train: Dataset({
        features: ['id', 'document', 'label', 'input_ids', 'attention_mask'],
        num_rows: 134995
    })
    validation: Dataset({
        features: ['id', 'document', 'label', 'input_ids', 'attention_mask'],
        num_rows: 15000
    })
    test: Dataset({
        features: ['id', 'document', 'label', 'input_ids', 'attention_mask'],
        num_rows: 49997
    })
})


In [34]:
from transformers import Trainer, TrainingArguments
from evaluate import load
import numpy as np

# 평가 지표 설정
metric = load('accuracy')

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    return metric.compute(predictions=predictions, references=labels)

# TrainingArguments 설정
training_args_nsmc = TrainingArguments(
    output_dir='transformers_nsmc',      # 저장 경로
    eval_strategy='epoch',               # 매 epoch 평가
    save_strategy='epoch',               # 매 epoch 저장
    learning_rate=2e-5,                  # 학습률
    per_device_train_batch_size=32,      # 배치 크기
    per_device_eval_batch_size=32,
    num_train_epochs=3,                  # 3 epoch
    weight_decay=0.01,
    load_best_model_at_end=True,         # 최고 성능 모델 자동 복원
    metric_for_best_model='accuracy',    # accuracy 기준
    fp16=True,                           # GPU 속도 향상
    save_total_limit=2,
    seed=42,
    report_to='none',
)

# Trainer 생성
trainer_nsmc = Trainer(
    model=klue_model,
    args=training_args_nsmc,
    train_dataset=nsmc_tokenized['train'],
    eval_dataset=nsmc_tokenized['validation'],
    compute_metrics=compute_metrics,
)

# 학습 시작!
trainer_nsmc.train()
print("학습 완료!")

Epoch,Training Loss,Validation Loss,Accuracy
1,0.250787,0.249686,0.903733
2,0.182433,0.252611,0.907333
3,0.119170,0.310523,0.905667


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

학습 완료!


## STEP 4. Fine-tuning으로 성능 향상 (목표: Accuracy 90% 이상)

### 성능 향상 전략
1. **데이터 전처리 개선** : 노이즈 제거, 특수문자 처리
2. **max_length 최적화** : 문장 길이 분포 분석 후 설정
3. **TrainingArguments 튜닝** : lr, batch_size, epoch 조정
4. **warmup 추가** : 학습 초반 안정화

In [35]:
import re

def clean_text(text):
    """노이즈 제거 전처리 함수"""
    if text is None:
        return ""
    # 특수문자 제거 (한글, 영문, 숫자, 공백만 유지)
    text = re.sub(r'[^가-힣a-zA-Z0-9\s]', ' ', text)
    # 연속 공백 제거
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# 전처리 적용
def preprocess(batch):
    batch['document'] = [clean_text(doc) for doc in batch['document']]
    return batch

nsmc_cleaned = nsmc_dataset.map(preprocess, batched=True)

# 빈 문장 제거
nsmc_cleaned = nsmc_cleaned.filter(lambda x: len(x['document']) > 0)

print("전처리 완료!")
print(f"train     : {len(nsmc_cleaned['train'])}개")
print(f"validation: {len(nsmc_cleaned['validation'])}개")
print(f"test      : {len(nsmc_cleaned['test'])}개")

Map:   0%|          | 0/134995 [00:00<?, ? examples/s]

Map:   0%|          | 0/15000 [00:00<?, ? examples/s]

Map:   0%|          | 0/49997 [00:00<?, ? examples/s]

Filter:   0%|          | 0/134995 [00:00<?, ? examples/s]

Filter:   0%|          | 0/15000 [00:00<?, ? examples/s]

Filter:   0%|          | 0/49997 [00:00<?, ? examples/s]

전처리 완료!
train     : 134442개
validation: 14936개
test      : 49782개


In [36]:
# 문장 길이 분포 확인 → max_length 최적화
lengths = [len(doc.split()) for doc in nsmc_cleaned['train']['document']]
print(f"평균 길이  : {np.mean(lengths):.1f}")
print(f"최대 길이  : {np.max(lengths)}")
print(f"75th 백분위: {np.percentile(lengths, 75):.1f}")
print(f"90th 백분위: {np.percentile(lengths, 90):.1f}")
print(f"95th 백분위: {np.percentile(lengths, 95):.1f}")
# 대부분의 문장을 커버하는 max_length 선택

평균 길이  : 7.8
최대 길이  : 42
75th 백분위: 9.0
90th 백분위: 17.0
95th 백분위: 24.0


In [37]:
# 문장 길이 분포 분석 결과 반영 → max_length=64로 설정
def tokenize_nsmc_v2(batch):
    return klue_tokenizer(
        batch['document'],
        truncation=True,
        padding='max_length',
        max_length=64,            # 길이 분포 분석 후 최적화
        return_token_type_ids=False,
    )

nsmc_tokenized_v2 = nsmc_cleaned.map(tokenize_nsmc_v2, batched=True)
print("개선된 토크나이징 완료!")

Map:   0%|          | 0/134442 [00:00<?, ? examples/s]

Map:   0%|          | 0/14936 [00:00<?, ? examples/s]

Map:   0%|          | 0/49782 [00:00<?, ? examples/s]

개선된 토크나이징 완료!


In [38]:
from transformers import TrainingArguments, Trainer
import gc
import torch

# 메모리 정리
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# 개선된 TrainingArguments
training_args_v2 = TrainingArguments(
    output_dir='transformers_nsmc_v2',
    eval_strategy='epoch',
    save_strategy='epoch',
    learning_rate=3e-5,               # lr 조정
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    num_train_epochs=5,               # epoch 늘리기
    weight_decay=0.01,
    warmup_ratio=0.1,                 # 초반 학습 안정화
    lr_scheduler_type='linear',
    load_best_model_at_end=True,
    metric_for_best_model='accuracy',
    greater_is_better=True,
    fp16=True,
    save_total_limit=2,
    seed=42,
    report_to='none',
)

# 모델 새로 로드
klue_model_v2 = AutoModelForSequenceClassification.from_pretrained(
    'klue/bert-base',
    num_labels=2
)

# Trainer 생성
trainer_nsmc_v2 = Trainer(
    model=klue_model_v2,
    args=training_args_v2,
    train_dataset=nsmc_tokenized_v2['train'],
    eval_dataset=nsmc_tokenized_v2['validation'],
    compute_metrics=compute_metrics,
)

# 학습 시작!
trainer_nsmc_v2.train()
print("개선된 학습 완료!")

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: klue/bert-base
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy
1,0.268168,0.272806,0.889060
2,0.202001,0.255495,0.905664
3,0.127879,0.328140,0.902651
4,0.084803,0.414317,0.900308
5,0.046584,0.505861,0.900040


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

개선된 학습 완료!


In [39]:
# Validation 평가
print("=== Validation 평가 ===")
val_results = trainer_nsmc_v2.evaluate()
print(val_results)

# Test 평가
print("\n=== Test 최종 평가 ===")
test_results = trainer_nsmc_v2.evaluate(nsmc_tokenized_v2['test'])
print(test_results)

=== Validation 평가 ===


Training Loss,Validation Loss,Epoch,Accuracy
0.046584,0.255495,5,0.905664


{'eval_loss': 0.25549545884132385, 'eval_accuracy': 0.9056641671130156}

=== Test 최종 평가 ===


Training Loss,Validation Loss,Epoch,Accuracy
0.046584,0.254662,5,0.903580


{'eval_loss': 0.25466227531433105, 'eval_accuracy': 0.9035796070868989}


## STEP 5. Bucketing & Dynamic Padding 적용

### Bucketing이란?
- 비슷한 길이의 문장끼리 같은 배치로 묶는 기법
- `group_by_length=True` 로 활성화

### Dynamic Padding이란?
- 배치 내 가장 긴 문장 길이에 맞게 패딩
- `padding='max_length'` → `padding=True` 로 변경
- 불필요한 패딩 최소화!

### 기대 효과
| 항목 | 일반 학습 | Bucketing |
|------|----------|-----------|
| 패딩 | 고정 max_length | 배치 내 최대 길이 |
| 연산량 | 많음 | 적음 ✅ |
| 학습 속도 | 느림 | 빠름 ✅ |
| 모델 성능 | 기준 | 비슷하거나 향상 |

In [40]:
from transformers import DataCollatorWithPadding
import time

# Dynamic Padding용 토크나이징
# padding 제거 → DataCollator가 배치마다 동적으로 패딩
def tokenize_nsmc_bucket(batch):
    return klue_tokenizer(
        batch['document'],
        truncation=True,      # 최대 길이 초과시 자르기
        max_length=128,       # 최대 길이 제한
        # padding 없음 → DataCollatorWithPadding이 동적으로 처리
    )

# 토크나이징 적용
nsmc_tokenized_bucket = nsmc_cleaned.map(
    tokenize_nsmc_bucket,
    batched=True
)
print("Bucketing용 토크나이징 완료!")

# Dynamic Padding DataCollator
data_collator = DataCollatorWithPadding(tokenizer=klue_tokenizer)

Map:   0%|          | 0/134442 [00:00<?, ? examples/s]

Map:   0%|          | 0/14936 [00:00<?, ? examples/s]

Map:   0%|          | 0/49782 [00:00<?, ? examples/s]

Bucketing용 토크나이징 완료!


In [41]:
# 메모리 정리
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# Bucketing용 TrainingArguments
# group_by_length 는 현재 버전에서 제거됨
# DataCollatorWithPadding으로 동적 패딩 구현
training_args_bucket = TrainingArguments(
    output_dir='transformers_nsmc_bucket',
    eval_strategy='epoch',
    save_strategy='epoch',
    learning_rate=3e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    num_train_epochs=5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    lr_scheduler_type='linear',
    load_best_model_at_end=True,
    metric_for_best_model='accuracy',
    greater_is_better=True,
    fp16=True,
    save_total_limit=2,
    seed=42,
    report_to='none',
)

# 모델 새로 로드
klue_model_bucket = AutoModelForSequenceClassification.from_pretrained(
    'klue/bert-base',
    num_labels=2
)

# 학습 시작 시간 기록
start_time = time.time()

# Trainer 생성 (DataCollatorWithPadding 추가!)
trainer_bucket = Trainer(
    model=klue_model_bucket,
    args=training_args_bucket,
    train_dataset=nsmc_tokenized_bucket['train'],
    eval_dataset=nsmc_tokenized_bucket['validation'],
    compute_metrics=compute_metrics,
    data_collator=data_collator,      # ✅ Dynamic Padding 적용
)

trainer_bucket.train()

# 학습 시간 기록
bucket_time = time.time() - start_time
print(f"Bucketing 학습 완료! 소요시간: {bucket_time:.1f}초")

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: klue/bert-base
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy
1,0.265483,0.267561,0.888324
2,0.200076,0.261800,0.904258
3,0.131357,0.326779,0.902049
4,0.083721,0.449619,0.900241
5,0.048178,0.503176,0.899839


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Bucketing 학습 완료! 소요시간: 2914.6초


In [42]:
# Bucketing 평가
print("=== Bucketing Validation 평가 ===")
bucket_val = trainer_bucket.evaluate()
print(bucket_val)

print("\n=== Bucketing Test 평가 ===")
bucket_test = trainer_bucket.evaluate(nsmc_tokenized_bucket['test'])
print(bucket_test)

# STEP 4 vs STEP 5 비교
print("\n" + "="*50)
print("📊 STEP 4 vs STEP 5 비교")
print("="*50)
print(f"{'항목':<20} {'STEP4(일반)':>15} {'STEP5(Bucketing)':>15}")
print("-"*50)
print(f"{'Validation Accuracy':<20} {val_results['eval_accuracy']:>15.4f} {bucket_val['eval_accuracy']:>15.4f}")
print(f"{'학습 소요시간(초)':<20} {'측정안함':>15} {bucket_time:>15.1f}")
print("="*50)

=== Bucketing Validation 평가 ===


Training Loss,Validation Loss,Epoch,Accuracy
0.048178,0.261800,5,0.904258


{'eval_loss': 0.2618003189563751, 'eval_accuracy': 0.9042581681842529}

=== Bucketing Test 평가 ===


Training Loss,Validation Loss,Epoch,Accuracy
0.048178,0.261656,5,0.903077


{'eval_loss': 0.2616555690765381, 'eval_accuracy': 0.9030774175404764}

📊 STEP 4 vs STEP 5 비교
항목                         STEP4(일반) STEP5(Bucketing)
--------------------------------------------------
Validation Accuracy           0.9057          0.9043
학습 소요시간(초)                      측정안함          2914.6


## 프로젝트 회고

### 잘한 점 👍

오늘 HuggingFace transformers framework를 활용하여 한국어 감성분석 모델을 직접 만들어보았습니다. NSMC 데이터셋을 로드하고, klue/bert-base 모델을 fine-tuning하여 Validation Accuracy 90% 이상이라는 목표를 달성했습니다. 또한 Bucketing과 Dynamic Padding을 적용하여 학습 효율화를 실험해보았고, 일반 학습과의 성능 및 속도를 비교 분석했습니다.

---

### 배운 점 📚

처음에는 HuggingFace의 Auto Class, Tokenizer, Trainer 등 각 구성요소가 낯설었지만, 직접 코드를 작성하고 오류를 해결하면서 framework의 전체적인 흐름을 이해하게 되었습니다. 특히 데이터 전처리 과정에서 결측값 제거, 특수문자 처리 등 데이터 품질이 모델 성능에 직접적인 영향을 미친다는 것을 실감했습니다. 또한 Bucketing이 단순히 속도만 빠른 것이 아니라, 패딩을 최소화하여 연산 효율을 높이는 원리를 이해할 수 있었습니다.

---

### 어려웠던 점 😅

라이브러리 버전 충돌 문제가 가장 힘들었습니다. pyarrow 버전 불일치, `group_by_length` 인자 제거, `tokenizer` 인자 제거 등 교재 코드와 현재 버전 사이의 차이를 하나씩 해결해나가는 과정이 많은 시간을 차지했습니다. 또한 NSMC 데이터셋을 `e9t/nsmc`로 로드하는 방식이 더 이상 지원되지 않아 GitHub raw 파일로 직접 로드하는 방식으로 우회해야 했습니다.

---

### 아쉬운 점 & 개선 방향 🔧

STEP4의 학습 소요시간을 측정하지 못해 STEP4와 STEP5의 속도 비교가 완전하지 않았습니다. 다음에는 두 학습 모두 시간을 측정하여 더 정확한 비교 분석을 하고 싶습니다. 또한 Accuracy 외에 F1 score도 함께 측정하면 더 풍부한 성능 분석이 가능할 것 같습니다.

---

### 느낀 점 💡

단순히 코드를 복붙하는 것이 아니라 오류를 직접 해결하고 버전 차이를 파악하는 과정에서 HuggingFace framework에 대한 실질적인 이해가 깊어졌습니다. 앞으로 다양한 모델과 데이터셋으로 더 많은 실험을 해보고 싶습니다! 😊